# Clinical AI Tool Agent — Demo

**EDUCATIONAL USE ONLY.** This is a teaching notebook. The agent's outputs are not medical advice.

Three runs demonstrate the agent's three primary code paths:
1. A pure-knowledge question (retriever → final answer, no tools).
2. A computation question (retriever → tool call → final answer).
3. A combined question (retriever → tool → re-prompt → final answer).

**Prerequisites:**
1. `ollama pull tinyllama`
2. `pip install -e .[dev]` from the repo root.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

from clinical_ai_tool_agent.agent import build_default_llm, run_pipeline_react
from clinical_ai_tool_agent.knowledge import build_retriever
from clinical_ai_tool_agent.tools import build_tools

llm = build_default_llm()
retriever = build_retriever()
tools = build_tools()
print(f'Loaded {len(tools)} tools:', [t.name for t in tools])

## Path 1 — knowledge retrieval only

In [ ]:
trace = run_pipeline_react(
    'What are the red flags for fever and cough? Education only.',
    llm=llm, retriever=retriever, tools=tools,
)
print('Tool calls:', trace.tool_calls)
print(f'\n--- Final ({trace.steps} step(s), {trace.latency_s}s) ---')
print(trace.final_answer)

## Path 2 — computation via tool calling

In [ ]:
trace = run_pipeline_react(
    'Given labs: Na 138, Cl 100, HCO3 22. Compute anion gap and outline interpretation.',
    llm=llm, retriever=retriever, tools=tools,
)
print('Tool calls:', trace.tool_calls)
print(f'\n--- Final ({trace.steps} step(s), {trace.latency_s}s) ---')
print(trace.final_answer)

## Path 3 — computation + retrieval combined

In [ ]:
trace = run_pipeline_react(
    'Patient height 1.68 m, weight 82 kg. Compute BMI and suggest next-step considerations.',
    llm=llm, retriever=retriever, tools=tools,
)
print('Tool calls:', trace.tool_calls)
print(f'\n--- Final ({trace.steps} step(s), {trace.latency_s}s) ---')
print(trace.final_answer)

## Inspecting raw model outputs

Every step's raw model output is preserved in `trace.raw_outputs`. Useful for understanding why a step branched the way it did.

In [ ]:
for i, raw in enumerate(trace.raw_outputs, 1):
    print(f'=== STEP {i} RAW MODEL OUTPUT ===')
    print(raw)
    print()